# <font color="blue">Análisis de señales ECG: frecuencia cardíaca y detección de arritmias con Python</font>
# <font color="red">Alfredo Díaz Claros</font>


## Objetivo de la práctica

En esta práctica trabajaremos con un **electrocardiograma (ECG) real** de la base de datos pública **LTAFDB** de PhysioNet. Al finalizar, usted podrá:

- Descargar y leer una señal ECG con la biblioteca `wfdb`.
- Graficar la señal y las anotaciones (latidos marcados por especialistas).
- Calcular la frecuencia cardíaca (FC) a partir de los intervalos RR.
- Clasificar cada latido como bradicardia, normal o taquicardia.
- Simular la lectura de un ECG en tiempo real, como en un monitor de pacientes.

**Requisitos:** una cuenta de Google y conexión a internet. Esta práctica **no requiere GPU**.

## Conceptos clave

- **ECG:** registro de la actividad eléctrica del corazón a lo largo del tiempo. En las gráficas, el eje horizontal es el tiempo y el vertical la amplitud, en milivoltios (mV).
- **Onda R:** el pico más alto de cada latido. Las anotaciones de la base de datos indican la posición de cada onda R.
- **Intervalo RR:** tiempo que transcurre entre dos ondas R consecutivas.
- **Frecuencia cardíaca instantánea (BPM):** $\text{BPM} = \dfrac{60}{RR\ \text{(en segundos)}}$.
- **Frecuencia de muestreo (fs):** número de muestras por segundo. Si fs = 250 Hz, cada muestra representa 4 milisegundos.

## ¿Cómo usar este cuaderno?

Ejecute las celdas **en orden**, de arriba hacia abajo, con `Shift + Enter` o con el botón ▶. Lea la explicación de cada paso **antes** de ejecutar la celda y trate de predecir qué va a pasar.

## Parte 1. Preparar el entorno y los datos

### Paso 1. Instalar la biblioteca `wfdb`

`wfdb` es la biblioteca oficial de PhysioNet para leer registros fisiológicos (archivos `.dat`, `.hea` y `.atr`). La opción `-q` oculta el detalle de la instalación.

In [ ]:
# INSTALAR BIBLIOTECA
!pip install wfdb -q

### Paso 2. Descargar el registro 23 de PhysioNet

Cada registro está formado por tres archivos:

- **`.dat`**: los valores de la señal ECG.
- **`.hea`**: la cabecera, con la frecuencia de muestreo y los nombres de los canales.
- **`.atr`**: las anotaciones, es decir, la posición de cada latido y su tipo.

`wget -q` descarga los archivos sin mostrar mensajes, y `-P` indica la carpeta de destino.

In [ ]:
# DESCARGAR DATOS ECG DE PHYSIONET

!mkdir -p ecg-data

!wget -q https://physionet.org/files/ltafdb/1.0.0/23.dat -P ecg-data/
!wget -q https://physionet.org/files/ltafdb/1.0.0/23.hea -P ecg-data/
!wget -q https://physionet.org/files/ltafdb/1.0.0/23.atr -P ecg-data/

## Parte 2. Importar bibliotecas

### Paso 3. Cargar las bibliotecas necesarias

- `wfdb`: leer señales y anotaciones.
- `numpy`: operaciones matemáticas con arreglos.
- `pandas`: tablas de datos.
- `matplotlib`: gráficos.

In [ ]:
# IMPORTAR BIBLIOTECAS

import wfdb
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

## Parte 3. Cargar y explorar la señal

### Paso 4. Definir la ruta del registro

La ruta `ecg-data/23` no incluye extensión. `wfdb` agrega automáticamente `.dat` y `.hea`.

In [ ]:
# RUTA DEL REGISTRO ECG

record_path = 'ecg-data/23'

### Paso 5. Leer las anotaciones (latidos marcados)

La función `wfdb.rdann` lee el archivo `.atr`. El parámetro `extension` indica el tipo de anotación: para el archivo `23.atr`, la extensión es `'atr'`.

El objeto `ann` contiene:
- `ann.sample`: posición (número de muestra) de cada anotación.
- `ann.symbol`: símbolo o tipo de latido.
- `ann.aux_note`: notas auxiliares, por ejemplo cambios de ritmo.

In [ ]:
# wfdb.rdann(record_name, extension, sampfrom=0, sampto=None,
#            shift_samps=False, pn_dir=None, return_label_elements=['symbol'],
#            summarize_labels=False)

# extension: texto con la extensión del archivo de anotaciones.
#            Por ejemplo, para '23.atr' la extensión es 'atr'.

ann = wfdb.rdann(record_path, 'atr')

### Paso 6. Leer el registro completo

`wfdb.rdrecord` lee la señal y devuelve un objeto `record` con sus atributos, como la frecuencia de muestreo y los nombres de los canales.

In [ ]:
# wfdb.rdrecord(record_name, sampfrom=0, sampto=None, channels=None, physical=True,
#               pn_dir=None, m2s=True, smooth_frames=True, ignore_skew=False,
#               return_res=64, force_channels=True, channel_names=None, warn_empty=False)

# Lee un registro WFDB y devuelve la señal y sus descriptores como atributos
# de un objeto Record o MultiRecord.

record = wfdb.rdrecord('ecg-data/23')

### Paso 7. Consultar la frecuencia de muestreo

Muestra cuántas muestras se toman por segundo.

In [ ]:
# Frecuencia de muestreo (muestras por segundo)

record.fs

### Paso 8. Consultar los canales con señal ECG

Un registro puede tener varios canales (derivaciones). Anote sus nombres; en esta práctica usaremos el **canal 0**.

In [ ]:
# Canales con señal ECG

record.sig_name

### Paso 9. Extraer el canal 0

`record.p_signal` es una matriz en la que cada fila es una muestra y cada columna es un canal. La expresión `[:, 0]` selecciona todas las filas de la primera columna.

In [ ]:
# Señal de ECG del canal 0

signal = record.p_signal[:, 0]

### Paso 10. Observar los valores de la señal

Son valores en milivoltios (mV). Jupyter muestra solo los extremos porque la lista es muy larga.

In [ ]:
# Valores de la señal (en mV)

signal

### Paso 11. Graficar la señal completa

Verá una gráfica muy densa. Es normal, porque el registro tiene millones de muestras.

In [ ]:
# Gráfica de la señal completa

plt.plot(signal)
plt.show()

### Paso 12. Contar las muestras

`len(signal)` indica el número total de muestras del registro.

In [ ]:
# Número total de muestras

len(signal)

### Paso 13. Calcular la duración en segundos

Duración = número de muestras / frecuencia de muestreo.

**Pregunta para pensar:** ¿por qué dividimos entre `fs`?

In [ ]:
# Duración en segundos

len(signal) / record.fs

### Paso 14. Convertir la duración a horas

In [ ]:
# Duración en horas

((len(signal) / record.fs) / 60) / 60

### Paso 15. Leer con `rdsamp`

`wfdb.rdsamp` es una alternativa más directa a `rdrecord`. Devuelve dos elementos: las señales (`signals`) y los campos descriptivos (`fields`).

In [ ]:
# wfdb.rdsamp(record_name, sampfrom=0, sampto=None, channels=None, pn_dir=None,
#             channel_names=None, warn_empty=False, return_res=64)

# Lee un registro WFDB y devuelve las señales físicas y algunos descriptores importantes.

signals, fields = wfdb.rdsamp(record_path)

### Paso 16. Revisar los descriptores (`fields`)

Contiene la información del registro: frecuencia de muestreo, unidades, nombres de los canales, entre otros.

In [ ]:
fields

### Paso 17. Revisar las señales (`signals`)

A diferencia de `signal`, esta variable contiene todos los canales: cada fila es una muestra y cada columna un canal.

In [ ]:
signals

## Parte 4. Visualizar la señal

### Paso 18. Graficar los primeros 20 segundos

Se crea un vector de tiempo (`tiempo`) dividiendo el número de muestra entre la frecuencia de muestreo. Así el eje horizontal muestra segundos y no muestras.

**Pregunta para pensar:** ¿en la gráfica puede distinguir las ondas P, QRS y T de un latido?

In [ ]:
# GRAFICAR ECG COMPLETO (PRIMEROS 20 SEGUNDOS)

tiempo = np.arange(len(signal)) / record.fs

plt.figure(figsize=(18, 5))

seconds_to_show = 20
samples = int(seconds_to_show * record.fs)

plt.plot(tiempo[:samples], signal[:samples])

plt.title("Señal ECG - Primeros 20 segundos")
plt.xlabel("Tiempo (s)")
plt.ylabel("Amplitud (mV)")
plt.grid(True)

plt.show()

### Paso 19. Hacer zoom en un fragmento corto

Para observar la forma de un latido, seleccionamos un fragmento de pocos segundos. El eje horizontal muestra el índice de muestra.

In [ ]:
# Fragmento representativo de la señal bruta

start = 7190174
end = 7190618
tiempo_parte = tiempo[start:end]
signal_part = signals[start:end, 0]

plt.figure(figsize=[15, 4])
plt.plot(signal_part, color='dimgray')

plt.grid()
plt.xlabel('Índice de muestra')
plt.ylabel('ECG [mV]')
plt.title('Muestra ECG')
plt.xlim(0, len(signal_part))
plt.ylim(signal_part.min(), signal_part.max() + 0.3)
plt.show()

## Parte 5. Anotaciones: los latidos marcados

Las anotaciones son la "respuesta correcta" de la base de datos: indican dónde está cada latido y qué tipo es.

En la base LTAFDB también se usan notas de ritmo en `aux_note`. Por ejemplo, `(AFIB` indica el inicio de **fibrilación auricular** y `(N` indica el regreso a ritmo normal.

### Paso 20. Crear una tabla con las anotaciones

Se construye un `DataFrame` con tres columnas: símbolo, nota auxiliar y posición.

In [ ]:
# `symbol` es la clase del latido, `aux_note` es la nota auxiliar y `sample` es el índice de la muestra en la señal.

df = pd.DataFrame({'symbol': ann.symbol, 'aux': ann.aux_note, 'position': ann.sample})
df.head()

### Paso 21. Consultar el significado de los símbolos

Muestra la tabla de símbolos de anotación de `wfdb`. Algunos ejemplos comunes:

- **N**: latido normal.
- **V**: contracción ventricular prematura.
- **A**: contracción auricular prematura.

In [ ]:
wfdb.io.show_ann_labels()

### Paso 22. Superponer las anotaciones sobre la señal

Los puntos verdes marcan la posición de los latidos anotados dentro del fragmento. Se resta `start` a cada posición para que coincida con el eje del fragmento.

In [ ]:
# Fragmento representativo de la señal bruta y sus anotaciones

start = 7190174
end = 7190618
signal_part = signals[start:end, 0]

plt.figure(figsize=[15, 4])
plt.plot(signal_part, color='dimgray')

# Selecciona las anotaciones que caen dentro del fragmento
ids = (df.position > start) & (df.position < end)
df_part = df[ids]

# Ajusta las posiciones para que coincidan con el fragmento
annotation_xs = df_part.position - start
annotation_ys = np.ones_like(annotation_xs) * 2.1
plt.plot(annotation_xs, annotation_ys, 'o', color='teal', ms=10)
plt.grid()
plt.xlabel('Índice de muestra')
plt.ylabel('ECG [mV]')
plt.title('Muestra ECG')
plt.xlim(0, len(signal_part))
plt.ylim(signal_part.min(), signal_part.max() + 1.666)
plt.show()

### Paso 23. Ver las anotaciones del fragmento en una tabla

In [ ]:
ids = (df.position > start) & (df.position < end)
df_part = df[ids]
df_part

### Paso 24. Graficar con `plot_wfdb` (primeros 10 000 muestras)

`wfdb.plot_wfdb` dibuja la señal con la cuadrícula de ECG y los símbolos de cada latido, de forma más simple que `matplotlib`.

In [ ]:
record1 = wfdb.rdrecord('ecg-data/23', sampto=10000)
annotation = wfdb.rdann('ecg-data/23', 'atr', sampto=10000)

wfdb.plot_wfdb(record=record1, annotation=annotation, plot_sym=True,
               time_units='seconds', title='23',
               figsize=(10, 4), ecg_grids='all')

### Paso 25. Graficar un segmento de 5 segundos con anotaciones

Se define el punto de inicio y la duración del segmento. Luego se leen la señal y las anotaciones solo de ese intervalo.

In [ ]:
# Definir la muestra inicial y la duración del segmento a graficar
start_sample = 7190174
duration_seconds = 5  # Graficar 5 segundos de datos

# Calcular la muestra final según la duración y la frecuencia de muestreo
fs = record.fs
end_sample = start_sample + int(duration_seconds * fs)

# Leer el registro y las anotaciones del intervalo
record_segment = wfdb.rdrecord(record_path, sampfrom=start_sample, sampto=end_sample)
annotation_segment = wfdb.rdann(record_path, 'atr', sampfrom=start_sample,
                                sampto=end_sample, shift_samps=True)

# Graficar el segmento con wfdb.plot_wfdb
wfdb.plot_wfdb(
    record=record_segment,
    annotation=annotation_segment,
    plot_sym=True,
    time_units='seconds',
    title=f'ECG Segment from Sample {start_sample} to {end_sample}',
    figsize=(15, 6),
    ecg_grids='all'
)
plt.show()

## Parte 6. Frecuencia cardíaca

### Paso 26. Calcular los intervalos RR

La función `ann2rr` calcula los intervalos RR, es decir, el número de muestras entre latidos consecutivos.

In [ ]:
# wfdb.processing.ann2rr(record_name, extension, pn_dir=None, start_time=None,
#                        stop_time=None, format=None, as_array=True)

# Obtiene la serie de intervalos RR a partir del archivo de anotaciones ECG.

import wfdb.processing
wfdb.processing.ann2rr(record_path, 'atr', as_array=True)

### Paso 27. Calcular la frecuencia cardíaca promedio

`calc_mean_hr` convierte los intervalos RR en latidos por minuto (BPM) y calcula el promedio. Para convertir muestras en segundos se usa `fs`.

In [ ]:
# wfdb.processing.calc_mean_hr(rr, fs=None, min_rr=None, max_rr=None, rr_units='samples')
# Calcula la frecuencia cardíaca promedio en latidos por minuto a partir de intervalos RR.
# Devuelve 0 si rr está vacío.

rr_intervals = wfdb.processing.ann2rr(record_path, 'atr', as_array=True)
mean_hr = wfdb.processing.calc_mean_hr(rr_intervals, fs=record.fs)

print("Frecuencia cardíaca promedio: ", round(mean_hr, 2))

### Paso 28. Calcular la frecuencia cardíaca instantánea en un fragmento de 10 segundos

Ahora calculamos la frecuencia **latido a latido**. La fórmula es:

$$\text{BPM}_i = \frac{60 \cdot fs}{RR_i \text{ (en muestras)}}$$

Cada intervalo RR se calcula como la diferencia entre dos posiciones de onda R consecutivas (`np.diff`).

In [ ]:
plot_start_sample = 7190174
plot_duration_seconds = 10
plot_fs = record.fs

plot_end_sample = plot_start_sample + int(plot_duration_seconds * plot_fs)

record_seg = wfdb.rdrecord(record_path, sampfrom=plot_start_sample, sampto=plot_end_sample)
ann_seg = wfdb.rdann(record_path, 'atr', sampfrom=plot_start_sample,
                     sampto=plot_end_sample, shift_samps=True)

ecg_signal_segment = record_seg.p_signal[:, 0]

r_peak_locations_segment = ann_seg.sample

# Intervalos RR instantáneos en el segmento (en muestras)
rr_intervals_segment_samples = np.diff(r_peak_locations_segment)

# Frecuencia instantánea (BPM). Cada valor corresponde al segundo pico R del intervalo.
if len(rr_intervals_segment_samples) > 0:
    instantaneous_hr_bpm = 60 * plot_fs / rr_intervals_segment_samples
else:
    instantaneous_hr_bpm = np.array([])  # Vacío si no hay suficientes picos R

time_axis_segment = np.arange(len(ecg_signal_segment)) / plot_fs

### Paso 29. Revisar los valores de BPM

Cada número es la frecuencia cardíaca calculada a partir de un par de latidos consecutivos.

In [ ]:
instantaneous_hr_bpm

### Paso 30. Graficar la señal con la frecuencia instantánea

Se dibujan los picos R (puntos rojos) y, sobre cada uno, su BPM.

In [ ]:
plt.figure(figsize=(15, 6))

plt.plot(time_axis_segment, ecg_signal_segment, color='dimgray', linewidth=1)

# Marcadores de picos R (tiempo relativo a la frecuencia de muestreo)
r_peak_times_segment = r_peak_locations_segment / plot_fs

valid_r_peak_indices = r_peak_locations_segment[r_peak_locations_segment < len(ecg_signal_segment)]
r_peak_values_segment = ecg_signal_segment[valid_r_peak_indices]
plt.plot(r_peak_times_segment[:len(valid_r_peak_indices)], r_peak_values_segment,
         'ro', markersize=6, label='R-picos')

# Etiquetas con el BPM de cada latido
if len(instantaneous_hr_bpm) > 0 and len(r_peak_times_segment) > 1:
    for i in range(len(instantaneous_hr_bpm)):

        hr_x_time = r_peak_times_segment[i + 1]
        hr_y_value = ecg_signal_segment[r_peak_locations_segment[i + 1]] + 0.1

        plt.text(hr_x_time, hr_y_value,
                 f'{instantaneous_hr_bpm[i]:.0f} BPM',
                 fontsize=8, color='blue', ha='center', va='bottom')

plt.title(f'Señal ECG con Frecuencia Cardíaca Instantánea (Muestras {plot_start_sample}-{plot_end_sample})')
plt.xlabel('Tiempo (segundos)')
plt.ylabel('Amplitud (mV)')
plt.grid(True)
plt.legend()
plt.tight_layout()
plt.show()

### Paso 31. Clasificar cada latido

Usamos criterios sencillos con fines didácticos:

- **Bradicardia:** menos de 60 BPM.
- **Normal:** entre 60 y 100 BPM.
- **Taquicardia:** más de 100 BPM.

> **Importante:** esta clasificación es solo educativa. Un diagnóstico real requiere la evaluación de un profesional de la salud.

In [ ]:
# Clasificación de arritmias (criterio didáctico por frecuencia)

arrhythmias_segment = []

for bpm in instantaneous_hr_bpm:
    if bpm < 60:
        arrhythmias_segment.append("Bradicardia")
    elif bpm > 100:
        arrhythmias_segment.append("Taquicardia")
    else:
        arrhythmias_segment.append("Normal")

# Tabla con los resultados
df_segment_arrhythmias = pd.DataFrame({
    "Instantaneous BPM": instantaneous_hr_bpm,
    "Clasificación": arrhythmias_segment
})

print("\nClasificación de arritmias para el segmento de ECG:")
print("--------------------------------------------------")
print(df_segment_arrhythmias)

summary_segment = df_segment_arrhythmias["Clasificación"].value_counts()

print("\nResumen de arritmias en el segmento:")
print(summary_segment)

### Paso 32. Graficar la clasificación con colores

Cada latido se colorea según su clasificación: verde (normal), rojo (taquicardia) y naranja (bradicardia).

In [ ]:
# Colores para cada clasificación
color_map = {
    "Bradicardia": "orange",
    "Normal": "green",
    "Taquicardia": "red"
}

plt.figure(figsize=(18, 6))

plt.plot(time_axis_segment, ecg_signal_segment, color='dimgray', linewidth=1)

r_peak_times_segment = r_peak_locations_segment / plot_fs

valid_r_peak_indices = r_peak_locations_segment[r_peak_locations_segment < len(ecg_signal_segment)]
r_peak_values_segment = ecg_signal_segment[valid_r_peak_indices]

if len(instantaneous_hr_bpm) > 0 and len(r_peak_times_segment) > 1:
    for i in range(len(instantaneous_hr_bpm)):
        classification = df_segment_arrhythmias['Clasificación'].iloc[i]
        hr_color = color_map.get(classification, 'black')

        hr_x_time = r_peak_times_segment[i + 1]
        hr_y_value_signal = ecg_signal_segment[r_peak_locations_segment[i + 1]]
        hr_y_value_text = hr_y_value_signal + 0.1

        plt.plot(hr_x_time, hr_y_value_signal, 'o', color=hr_color, markersize=7, alpha=0.8)

        plt.text(hr_x_time, hr_y_value_text,
                 f'{instantaneous_hr_bpm[i]:.0f} BPM ({classification})',
                 fontsize=8, color=hr_color, ha='center', va='bottom')

plt.title(f'Señal ECG con Frecuencia Cardíaca Instantánea y Clasificación de Arritmias\n(Muestras {plot_start_sample}-{plot_end_sample})')
plt.xlabel('Tiempo (segundos)')
plt.ylabel('Amplitud (mV)')
plt.grid(True)
plt.legend(handles=[
    plt.Line2D([0], [0], marker='o', color='w', label='Normal', markerfacecolor='green', markersize=10),
    plt.Line2D([0], [0], marker='o', color='w', label='Taquicardia', markerfacecolor='red', markersize=10),
    plt.Line2D([0], [0], marker='o', color='w', label='Bradicardia', markerfacecolor='orange', markersize=10)],
    title='Clasificación de HR')
plt.tight_layout()
plt.show()

## Parte 7. Simulación de monitoreo en tiempo real

En esta última parte simulamos un monitor de pacientes: la señal se muestra en bloques de 5 segundos que avanzan uno tras otro, mientras se calcula la frecuencia y la clasificación en cada bloque.

### Paso 33. Ejecutar la simulación

**Ajuste realizado para clase:** la versión original procesaba todo el registro, que dura muchas horas, y tardaría demasiado. Ahora la simulación se limita a `duracion_simulacion = 60` segundos. Puede aumentar este valor si desea una simulación más larga.

Para detener la simulación antes de tiempo, use el botón **Interrumpir ejecución** (■) de Colab.

In [ ]:
# SIMULACIÓN DE ECG EN TIEMPO REAL CON BPM Y CLASIFICACIÓN

import time
from IPython.display import display, clear_output

# --- Configuración (reutiliza variables de celdas anteriores) ---
fs = record.fs                 # Frecuencia de muestreo
full_signal = signal           # Señal ECG completa (canal 0)

# Parámetros de la simulación
segment_seconds = 5                                  # Duración de cada segmento mostrado (s)
samples_per_segment = int(segment_seconds * fs)
duracion_simulacion = 60                             # Segundos de ECG a simular (ajustable)
total_signal_length = min(len(full_signal), int(duracion_simulacion * fs))

# Colores para cada clasificación
color_map = {
    "Bradicardia": "orange",
    "Normal": "green",
    "Taquicardia": "red"
}

# --- Configurar la gráfica interactiva ---
plt.ion()  # Activa el modo interactivo para actualizar la gráfica
fig, ax = plt.subplots(figsize=(18, 6))

ax.set_title('Simulación de Señal ECG en Tiempo Real con BPM y Arritmias')
ax.set_xlabel('Tiempo (segundos)')
ax.set_ylabel('Amplitud (mV)')
ax.grid(True)

legend_handles = [
    plt.Line2D([0], [0], marker='o', color='w', label='Normal', markerfacecolor='green', markersize=10),
    plt.Line2D([0], [0], marker='o', color='w', label='Taquicardia', markerfacecolor='red', markersize=10),
    plt.Line2D([0], [0], marker='o', color='w', label='Bradicardia', markerfacecolor='orange', markersize=10)
]
ax.legend(handles=legend_handles, title='Clasificación de HR', loc='upper right')

# Límites fijos del eje vertical, estimados con el primer minuto de datos
min_val = np.min(full_signal[0:min(len(full_signal), fs * 60)])
max_val = np.max(full_signal[0:min(len(full_signal), fs * 60)])
ax.set_ylim(min_val - 0.2, max_val + 0.5)

# --- Simulación de transmisión en tiempo real ---
current_start_sample = 0
plot_update_interval_seconds = 0.5  # Pausa entre actualizaciones (no es la duración de los datos)

print("Iniciando simulación de ECG en tiempo real. Presione 'Interrumpir ejecución' para detenerla.")

try:
    while current_start_sample < total_signal_length:
        current_end_sample = min(current_start_sample + samples_per_segment, total_signal_length)

        # Si el segmento está incompleto, termina (excepto el primero)
        if current_end_sample - current_start_sample < samples_per_segment and current_start_sample != 0:
            break

        # Segmento actual de la señal
        ecg_segment = full_signal[current_start_sample:current_end_sample]
        # Tiempo absoluto para el eje X del segmento
        time_segment = np.arange(len(ecg_segment)) / fs + (current_start_sample / fs)

        # Anotaciones del segmento, con posiciones relativas al inicio del segmento
        ann_seg = wfdb.rdann(record_path, 'atr', sampfrom=current_start_sample,
                             sampto=current_end_sample, shift_samps=True)
        r_peak_locations_segment = ann_seg.sample

        # Cálculo de BPM y clasificación
        instantaneous_hr_bpm = np.array([])
        arrhythmias_segment = []

        if len(r_peak_locations_segment) >= 2:  # Se necesitan al menos dos picos R
            rr_intervals_segment_samples = np.diff(r_peak_locations_segment)
            rr_intervals_segment_samples = rr_intervals_segment_samples[rr_intervals_segment_samples > 0]

            if len(rr_intervals_segment_samples) > 0:
                instantaneous_hr_bpm = 60 * fs / rr_intervals_segment_samples

                for bpm_val in instantaneous_hr_bpm:
                    if bpm_val < 60:
                        arrhythmias_segment.append("Bradicardia")
                    elif bpm_val > 100:
                        arrhythmias_segment.append("Taquicardia")
                    else:
                        arrhythmias_segment.append("Normal")
            else:
                instantaneous_hr_bpm = np.array([])
                arrhythmias_segment = []

        # --- Actualizar la gráfica ---
        ax.clear()
        ax.plot(time_segment, ecg_segment, color='dimgray', linewidth=1)

        # Marcadores de picos R, con color según la clasificación
        # Se grafica a partir del segundo pico R de cada intervalo
        if len(instantaneous_hr_bpm) > 0 and len(r_peak_locations_segment) >= 2:
            r_peak_times_for_plotting = (r_peak_locations_segment[1:] / fs) + (current_start_sample / fs)
            r_peak_values_for_plotting = ecg_segment[r_peak_locations_segment[1:]]

            for i in range(len(instantaneous_hr_bpm)):
                if i < len(arrhythmias_segment):
                    classification = arrhythmias_segment[i]
                    hr_color = color_map.get(classification, 'black')

                    hr_x_time = r_peak_times_for_plotting[i]
                    hr_y_value_signal = r_peak_values_for_plotting[i]
                    hr_y_value_text = hr_y_value_signal + 0.1

                    ax.plot(hr_x_time, hr_y_value_signal, 'o', color=hr_color, markersize=7, alpha=0.8)

                    ax.text(hr_x_time, hr_y_value_text,
                            f'{instantaneous_hr_bpm[i]:.0f} BPM ({classification})',
                            fontsize=8, color=hr_color, ha='center', va='bottom')

        ax.set_title('Simulación de Señal ECG en Tiempo Real con BPM y Arritmias\n' +
                     f'Tiempo: {time_segment[0]:.2f}s - {time_segment[-1]:.2f}s')
        ax.set_xlabel('Tiempo (segundos)')
        ax.set_ylabel('Amplitud (mV)')
        ax.grid(True)
        ax.set_xlim(time_segment[0], time_segment[-1])
        ax.set_ylim(min_val - 0.2, max_val + 0.5)
        ax.legend(handles=legend_handles, title='Clasificación de HR', loc='upper right')

        # Mostrar la gráfica actualizada
        clear_output(wait=True)
        display(fig)

        # Avanzar al siguiente segmento
        current_start_sample += samples_per_segment
        time.sleep(plot_update_interval_seconds)  # Pausa para simular el tiempo real

except KeyboardInterrupt:
    print("\nSimulación detenida por el usuario.")
finally:
    plt.ioff()
    plt.close(fig)
    print("Fin de la simulación.")

## Preguntas para la discusión en clase

1. En el Paso 13 dividimos el número de muestras entre `fs`. ¿Qué unidad tiene el resultado y por qué?
2. Si un intervalo RR pasa de 1 segundo a 0,5 segundos, ¿cómo cambia el BPM? Compruébelo con la fórmula del Paso 28.
3. Cambie el umbral de taquicardia de 100 a 110 BPM en el Paso 31. ¿Cuántos latidos cambian de clasificación?
4. ¿En qué se diferencia la frecuencia promedio (Paso 27) de la frecuencia instantánea (Paso 28)? ¿Cuál describe mejor los cambios del ritmo?
5. Si el registro tiene más de un canal (Paso 8), cambie `[:, 0]` por `[:, 1]` en el Paso 9 y observe si la señal se ve diferente.
6. Aumente `duracion_simulacion` en el Paso 33. ¿Qué observa cuando aparecen los segmentos con anotaciones de fibrilación auricular?

## Limitaciones

Esta práctica tiene fines didácticos. La clasificación por frecuencia es una simplificación: un análisis clínico real considera la forma de las ondas, el ritmo, el contexto del paciente y la valoración de un profesional de la salud.